In [1]:
from pathlib import Path

Path("exports/final_comparison/panels").mkdir(parents=True, exist_ok=True)
Path("exports/final_comparison/tables").mkdir(parents=True, exist_ok=True)
Path("exports/final_comparison/figures").mkdir(parents=True, exist_ok=True)

print("Final comparison folders created.")

Final comparison folders created.


In [15]:
# =============================================================================
# STEP 1 — BUILD COMPARISON PANELS (ROBUST VERSION)
# =============================================================================

import pandas as pd
from pathlib import Path

# Paths
BASE = Path(".")
PANEL_DIR = BASE / "exports/final_comparison/panels"
TABLE_DIR = BASE / "exports/final_comparison/tables"

PANEL_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Model file locations
FILE_MAP = {

"unemployment":{

"sarimax":"exports/test_predictions/sarimax_unemployment.csv",
"tbats_prophet":"exports/test_predictions/tbats_prophet_unemployment.csv",
"var":"exports/test_predictions/var_unemployment.csv",
"bilstm":"exports/test_predictions/bilstm_unemployment.csv",
"ensemble":"exports/ensemble/ensemble_unemployment.csv",
"rf_hybrid":"exports/rf_model/rf_unemployment.csv",
"chronos_uni":"exports/chronos/chronos_univariate_unemployment.csv",
"chronos_mv":"exports/chronos/chronos_multivariate_serialized_unemployment.csv"
},

"vacancies":{

"sarimax":"exports/test_predictions/sarimax_vacancies.csv",
"tbats_prophet":"exports/test_predictions/tbats_prophet_vacancies.csv",
"var":"exports/test_predictions/var_vacancies.csv",
"bilstm":"exports/test_predictions/bilstm_vacancies.csv",
"ensemble":"exports/ensemble/ensemble_vacancies.csv",
"rf_hybrid":"exports/rf_model/rf_vacancies.csv",
"chronos_uni":"exports/chronos/chronos_univariate_vacancies.csv",
"chronos_mv":"exports/chronos/chronos_multivariate_serialized_vacancies.csv"
},

"theta":{

"sarimax":"exports/test_predictions/sarimax_theta.csv",
"tbats_prophet":"exports/test_predictions/tbats_prophet_theta.csv",
"var":"exports/test_predictions/var_theta.csv",
"bilstm":"exports/test_predictions/bilstm_theta.csv",
"ensemble":"exports/ensemble/ensemble_theta.csv",
"rf_hybrid":"exports/rf_model/rf_theta.csv",
"chronos_uni":"exports/chronos/chronos_univariate_theta.csv",
"chronos_mv":"exports/chronos/chronos_multivariate_serialized_theta.csv"
}

}

# Standardize function
def load_file(path):

    df = pd.read_csv(path)

    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

    cols = {c.lower():c for c in df.columns}

    month = cols.get("month") or cols.get("forecast_date") or cols.get("date")
    actual = cols.get("actual") or cols.get("actual_value")

    pred = None
    for p in ["prediction","predicted_value","forecast","y_pred",
              "chronos_uni_pred","chronos_mv_pred","ensemble_pred",
              "final_forecast"]:
        if p in cols:
            pred = cols[p]
            break

    out = df[[month,actual,pred]].copy()

    out.columns=["month","actual","prediction"]

    out["month"]=pd.to_datetime(out["month"])

    return out

# Build panels
for target,models in FILE_MAP.items():

    merged=None

    for name,path in models.items():

        try:
            df=load_file(path)
        except:
            continue

        df=df.rename(columns={"prediction":f"{name}_pred"})

        if merged is None:
            merged=df
        else:
            merged=merged.merge(df[["month",f"{name}_pred"]],
                                on="month",
                                how="outer")

    merged=merged.sort_values("month")

    merged=merged.dropna(subset=["actual"])

    merged.to_csv(PANEL_DIR/f"comparison_{target}.csv",index=False)

    print("Panel created:",target,"rows:",len(merged))

Panel created: unemployment rows: 300
Panel created: vacancies rows: 300
Panel created: theta rows: 300


In [16]:
# =============================================================================
# STEP 2 — METRICS AND MODEL RANKING
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

PANEL_DIR = Path("exports/final_comparison/panels")
TABLE_DIR = Path("exports/final_comparison/tables")

targets=["unemployment","vacancies","theta"]

rows=[]

def rmse(a,p):
    m=~(np.isnan(a)|np.isnan(p))
    a,p=a[m],p[m]
    return np.sqrt(np.mean((a-p)**2)) if len(a)>0 else np.nan

def mae(a,p):
    m=~(np.isnan(a)|np.isnan(p))
    a,p=a[m],p[m]
    return np.mean(np.abs(a-p)) if len(a)>0 else np.nan

def mape(a,p):
    m=~(np.isnan(a)|np.isnan(p)|(a==0))
    a,p=a[m],p[m]
    return np.mean(np.abs((a-p)/a))*100 if len(a)>0 else np.nan

def smape(a,p):
    m=~(np.isnan(a)|np.isnan(p))
    a,p=a[m],p[m]
    return np.mean(2*np.abs(p-a)/(np.abs(a)+np.abs(p)))*100 if len(a)>0 else np.nan

def mase(a,p):
    m=~(np.isnan(a)|np.isnan(p))
    a,p=a[m],p[m]
    if len(a)<2: return np.nan
    naive=np.mean(np.abs(np.diff(a)))
    if naive==0: return np.nan
    return np.mean(np.abs(a-p))/naive

for t in targets:

    df=pd.read_csv(PANEL_DIR/f"comparison_{t}.csv")

    actual=df["actual"].values

    models=[c for c in df.columns if c.endswith("_pred")]

    for m in models:

        pred=df[m].values

        rows.append({
        "target":t,
        "model":m.replace("_pred",""),
        "RMSE":rmse(actual,pred),
        "MAE":mae(actual,pred),
        "MAPE":mape(actual,pred),
        "sMAPE":smape(actual,pred),
        "MASE":mase(actual,pred)
        })

metrics=pd.DataFrame(rows)

metrics["RMSE_rank"]=metrics.groupby("target")["RMSE"].rank()
metrics["MAE_rank"]=metrics.groupby("target")["MAE"].rank()
metrics["MASE_rank"]=metrics.groupby("target")["MASE"].rank()

metrics=metrics.sort_values(["target","RMSE"])

display(metrics)

metrics.to_csv(TABLE_DIR/"final_metrics_summary.csv",index=False)

print("Metrics saved.")

,target,model,RMSE,MAE,MAPE,sMAPE,MASE,RMSE_rank,MAE_rank,MASE_rank
20,theta,ensemble,9.033112e-02,8.011127e-02,19.845213,17.624203,24.031848,1.0,2.0,2.0
22,theta,chronos_uni,1.042288e-01,9.974828e-02,30.104710,36.048409,51.550457,2.5,3.5,4.5
23,theta,chronos_mv,1.042288e-01,9.974828e-02,30.104710,36.048409,51.550457,2.5,3.5,4.5
19,theta,bilstm,1.118855e-01,7.390648e-02,14.039006,15.815396,22.170529,4.0,1.0,1.0
18,theta,var,1.634218e-01,1.428730e-01,39.086785,30.638221,42.859152,5.0,5.0,3.0
16,theta,sarimax,2.871375e-01,2.677607e-01,70.950768,49.432117,80.323094,6.0,6.0,6.0
17,theta,tbats_prophet,4.661845e-01,4.400419e-01,115.795765,68.640945,132.004148,7.0,7.0,7.0
21,theta,rf_hybrid,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,unemployment,chronos_uni,7.546977e+04,5.728102e+04,3.667342,3.782792,12.637932,1.5,1.5,1.5
7,unemployment,chronos_mv,7.546977e+04,5.728102e+04,3.667342,3.782792,12.637932,1.5,1.5,1.5


Metrics saved.


In [17]:
# =============================================================================
# STEP 3 — DIEBOLD–MARIANO TESTS + WINNER SELECTION
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations
from scipy.stats import norm

PANEL_DIR = Path("exports/final_comparison/panels")
TABLE_DIR = Path("exports/final_comparison/tables")
TABLE_DIR.mkdir(parents=True, exist_ok=True)

targets = ["unemployment","vacancies","theta"]

# ---------------------------------------------------------------------
# Diebold–Mariano test (squared error loss)
# ---------------------------------------------------------------------
def diebold_mariano(actual, pred1, pred2):
    e1 = actual - pred1
    e2 = actual - pred2

    d = (e1**2) - (e2**2)
    d = d[~np.isnan(d)]

    T = len(d)
    if T < 5:
        return np.nan, np.nan

    mean_d = np.mean(d)
    var_d = np.var(d, ddof=1)

    dm_stat = mean_d / np.sqrt(var_d / T)
    p_value = 2 * (1 - norm.cdf(abs(dm_stat)))

    return dm_stat, p_value

dm_rows = []

for target in targets:

    df = pd.read_csv(PANEL_DIR / f"comparison_{target}.csv")

    if df.empty:
        continue

    actual = pd.to_numeric(df["actual"], errors="coerce").values

    model_cols = [c for c in df.columns if c.endswith("_pred")]

    for m1, m2 in combinations(model_cols, 2):

        p1 = pd.to_numeric(df[m1], errors="coerce").values
        p2 = pd.to_numeric(df[m2], errors="coerce").values

        dm, p = diebold_mariano(actual, p1, p2)

        dm_rows.append({
            "target":target,
            "model_1":m1.replace("_pred",""),
            "model_2":m2.replace("_pred",""),
            "DM_stat":dm,
            "p_value":p,
            "significant_5pct":p < 0.05 if not np.isnan(p) else False
        })

dm_df = pd.DataFrame(dm_rows)

display(dm_df)

dm_df.to_csv(TABLE_DIR/"dm_test_results.csv",index=False)

print("DM results saved.")

# ---------------------------------------------------------------------
# Winner selection
# ---------------------------------------------------------------------

metrics = pd.read_csv(TABLE_DIR/"final_metrics_summary.csv")

winner_rows = []

for target in targets:

    temp = metrics[metrics["target"] == target].copy()

    if temp.empty:
        continue

    temp = temp.sort_values("MASE")

    winner = temp.iloc[0]

    winner_rows.append({
        "target":target,
        "winner_model":winner["model"],
        "RMSE":winner["RMSE"],
        "MAE":winner["MAE"],
        "MASE":winner["MASE"]
    })

winner_df = pd.DataFrame(winner_rows)

display(winner_df)

winner_df.to_csv(TABLE_DIR/"final_winner_by_target.csv",index=False)

print("Final winners saved.")

C:\Users\Anurodh\AppData\Local\Temp\ipykernel_22880\3064816027.py:34: RuntimeWarning: invalid value encountered in scalar divide
  dm_stat = mean_d / np.sqrt(var_d / T)
C:\Users\Anurodh\AppData\Local\Temp\ipykernel_22880\3064816027.py:34: RuntimeWarning: invalid value encountered in scalar divide
  dm_stat = mean_d / np.sqrt(var_d / T)
C:\Users\Anurodh\AppData\Local\Temp\ipykernel_22880\3064816027.py:34: RuntimeWarning: invalid value encountered in scalar divide
  dm_stat = mean_d / np.sqrt(var_d / T)


,target,model_1,model_2,DM_stat,p_value,significant_5pct
0,unemployment,sarimax,tbats_prophet,40.156976,0.000000,True
1,unemployment,sarimax,var,30.928816,0.000000,True
2,unemployment,sarimax,bilstm,-103.286694,0.000000,True
3,unemployment,sarimax,ensemble,24.376494,0.000000,True
4,unemployment,sarimax,rf_hybrid,NaN,NaN,False
...,...,...,...,...,...,...
79,theta,ensemble,chronos_uni,-4.216429,0.000025,True
80,theta,ensemble,chronos_mv,-4.216429,0.000025,True
81,theta,rf_hybrid,chronos_uni,NaN,NaN,False
82,theta,rf_hybrid,chronos_mv,NaN,NaN,False


DM results saved.


,target,winner_model,RMSE,MAE,MASE
0,unemployment,chronos_uni,75469.765341,57281.017628,12.637932
1,vacancies,ensemble,75778.028403,65604.353978,17.814237
2,theta,bilstm,0.111885,0.073906,22.170529


Final winners saved.


In [18]:
# =============================================================================
# FINAL MODEL SELECTION (PRODUCTION MODEL)
# Chronos is benchmark only and cannot be selected
# =============================================================================

import pandas as pd
from pathlib import Path

TABLE_DIR = Path("exports/final_comparison/tables")

metrics = pd.read_csv(TABLE_DIR/"final_metrics_summary.csv")

# Chronos is benchmark only
metrics = metrics[~metrics["model"].isin(["chronos_uni","chronos_mv"])]

winner_rows = []

for target in metrics["target"].unique():

    temp = metrics[metrics["target"]==target].copy()

    # primary decision metric
    temp = temp.sort_values(["MASE","RMSE"])

    winner = temp.iloc[0]

    winner_rows.append({
        "target":target,
        "production_model":winner["model"],
        "RMSE":winner["RMSE"],
        "MAE":winner["MAE"],
        "MASE":winner["MASE"]
    })

winner_df = pd.DataFrame(winner_rows)

display(winner_df)

winner_df.to_csv(
    TABLE_DIR/"final_production_models.csv",
    index=False
)

print("Production models locked.")

,target,production_model,RMSE,MAE,MASE
0,theta,bilstm,0.111885,0.073906,22.170529
1,unemployment,ensemble,114033.697957,96540.545563,28.233199
2,vacancies,ensemble,75778.028403,65604.353978,17.814237


Production models locked.
